In [ ]:
import os, sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("utils"), "..")))
from module.utils import *


from module.prompt import *
from langchain_tavily import TavilySearch
from langgraph.prebuilt import create_react_agent


from typing import Annotated, List
from langchain_core.tools import tool


from langchain_community.document_loaders import WebBaseLoader


import bs4


from langchain_openai import OpenAIEmbeddings


from langchain.retrievers.document_compressors import CrossEncoderReranker


from langchain.retrievers import ContextualCompressionRetriever


from langchain_community.document_transformers import LongContextReorder


from langchain.retrievers import BM25Retriever, EnsembleRetriever

from operator import itemgetter


from langchain_core.runnables import RunnableLambda

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [19]:
def get_prompt_web_search():
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """ 
                # Instructions:
                You are a professional web search agent.
                It is important to carefully analyze and fully understand the requirements, which can lead to high-quality answers.
                Follow these steps strictly for optimal information retrieval and response generation:

                # Steps:
                1. If the requirements are complex or multifaceted, divide them into keyword groups and perform separate searches for each group.
                
                2. Use the keywords 'tavily_search_tool' for each group to gather information.

                3. Combine all the collected information and solve the problem

                4. Finally, it produces clear, concise, and complete answers in Korean.

                # Important:
                Follow this workflow sequentially: 
                Step 1 : (User Query Analysis and Segmentation) -> 
                Step 2 : (Tavily_search_tool) -> 
                Step 3 : (Merge Information Collected) -> 
                Step 4 : (final Korean answer)
                """,
            ),
            ("human", "{messages}"),
            ("placeholder", "{agent_scratchpad}"),
        ]
    )


def get_prompt():
    template = """당신은 유용한 AI 어시스턴트입니다. 사용자의 질의에 대해 친절하고 정확하게 답변해야 합니다.
    You are a helpful AI assistant, you'll need to answer users' queries in a friendly and accurate manner.
    모든 대답은 반드시 한국말로 대답해주세요.
    
    # User Question :
    {question}
    
    # Context :
    {context}

    # Output Format :
    1. Title:
    2. Content:
    3. Source:
    """
    prompt = PromptTemplate(template=template, input_variables=["context", "question"])
    return prompt

In [20]:
def crawling(urls):
    docs = []
    if len(urls) > 0:
        web_loader = WebBaseLoader(
            web_paths=(urls),
            bs_kwargs=dict(parse_only=bs4.SoupStrainer("body")),
        )
        results = web_loader.load()
        docs = results

    return docs


def get_split_docs(news_doc):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
    return text_splitter.split_documents(news_doc)


def get_retriever(split_docs, model="text-embedding-3-small"):
    embeddings = OpenAIEmbeddings(model=model)
    db = FAISS.from_documents(documents=split_docs, embedding=embeddings)
    return db.as_retriever(
        search_type="mmr", search_kwargs={"k": 5, "lambda_mult": 0.25, "fetch_k": 10}
    )


def get_bm25_retriever(split_docs):
    bm25_retriever = BM25Retriever.from_documents(split_docs)
    bm25_retriever.k = 5  # BM25Retriever의 검색 결과 개수를 1로 설정합니다.
    return bm25_retriever


def get_reranker(esenmble_retriever, user_request):
    docs = esenmble_retriever.invoke(user_request)
    documents_text = [doc.page_content for doc in docs]
    # print(documents_text)
    co = get_cohere_raranker()
    response = co.rerank(
        query=user_request,
        documents=documents_text,
        model="rerank-multilingual-v3.0",
        top_n=6,  # 상위 3개 결과만 리랭킹 반환
    )
    reranked_docs = []
    for result in response.results:
        reranked_docs.append(
            {
                # ** 연산자는 딕셔너리의 키-값 쌍을 풀어헤쳐 새로운 딕셔너리에 넣을 때 사용합니다.
                # 예를 들어, a = {'x':1, 'y':2}, b = {'z':3} 라면 {**a, **b}는 {'x':1, 'y':2, 'z':3}가 됩니다.
                **docs[result.index].dict(),
                "rerank_score": result.relevance_score,
            }
        )
    return reranked_docs


def reorder_documents(docs):
    # 재정렬
    reordering = LongContextReorder()
    reordered_docs = reordering.transform_documents(docs)
    return reordered_docs


def get_esenmble_retriever(retriever1, retriever2):
    ensemble_retriever = EnsembleRetriever(
        retrievers=[retriever1, retriever2],
        weights=[0.6, 0.4],  # 각 리트리버의 가중치를 설정합니다.
        k=6,  # 최종적으로 반환할 문서의 개수를 설정합니다.
    )
    return ensemble_retriever

In [21]:
# 웹검색시 키워드검색과 문장 검색 효율성 논문 찾아보기
@tool
def tavily_search_tool(
    user_request: Annotated[str, "user_request sentence"],
) -> list[str]:
    """
    사용자의 요청을 분석하여 실제 검색을 수행하기위한 도구

    Args:
    user_request(str) : 사용자의 요구사항

    Return :
    any
    """
    tool = TavilySearch(
        max_results=5,
        topic="general",
        # include_domains=["kr.wikipedia.org", "news.naver.com","entertain.naver.com","sports.naver.com"],
        exclude_domains=["youtube.com", "youtubekids.com"],
    )
    llm = get_gpt()
    prompt = get_prompt()
    response = tool.invoke({"query": user_request})
    urls = [result["url"] for result in response["results"]]
    docs = crawling(urls)
    split_docs = get_split_docs(docs)
    faiss_retriever = get_retriever(split_docs)
    bm25_retriever = get_bm25_retriever(split_docs)
    esenmble_retriever = get_esenmble_retriever(faiss_retriever, bm25_retriever)
    reranker_docs = get_reranker(esenmble_retriever, user_request)
    reorder_docs = reorder_documents(reranker_docs)
    # return reorder_docs
    chain = prompt | llm
    return chain.invoke({"question": user_request, "context": reorder_docs})

In [22]:
# question = "마이클 조던과 생일이 같은 유명인 3명 찾아줘"
question = "마이클 조던 생일"
# question = "마이클 조던이름 신문기사 3개"
tavily_search_tool(question)

C:\Users\ansgy\AppData\Local\Temp\ipykernel_21164\3370721906.py:50: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  **docs[result.index].dict(),


AIMessage(content='1. Title:  \n마이클 조던의 생일\n\n2. Content:  \n마이클 조던은 1963년 2월 17일에 태어났습니다.\n\n3. Source:  \nhttps://ko.wikipedia.org/wiki/마이클_조던', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 3737, 'total_tokens': 3796, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c064fdde7c', 'id': 'chatcmpl-CQlRvYZDfHuMytyGa1LkIIQm3ADGk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--eb7d4ca9-9688-4d2d-87c5-3402193a9fe2-0', usage_metadata={'input_tokens': 3737, 'output_tokens': 59, 'total_tokens': 3796, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [19]:
prompt = get_prompt_web_search()
llm = get_gemini()
agent = create_react_agent(prompt=prompt, model=llm, tools=[tavily_search_tool])

question = "마이클 조던과 생일이 같은 유명인 3명 찾아줘"
config = get_runnable_config(recursion_limit=10, thread_id=get_random_uuid())
inputs = {"messages": question}
for a in agent.stream(input=inputs, config=config, stream_mode="values"):
    print(a)

{'messages': [HumanMessage(content='마이클 조던과 생일이 같은 유명인 3명 찾아줘', additional_kwargs={}, response_metadata={}, id='b09b35d8-4544-4085-bed0-9c29b351ac75')]}
{'messages': [HumanMessage(content='마이클 조던과 생일이 같은 유명인 3명 찾아줘', additional_kwargs={}, response_metadata={}, id='b09b35d8-4544-4085-bed0-9c29b351ac75'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"user_request": "\\ub9c8\\uc774\\ud074 \\uc870\\ub358 \\uc0dd\\uc77c"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--95a1c61a-85aa-4b89-8cd5-d08b3e4b03f7-0', tool_calls=[{'name': 'tavily_search_tool', 'args': {'user_request': '마이클 조던 생일'}, 'id': '899518a6-b612-4beb-92f6-52eb18bea104', 'type': 'tool_call'}], usage_metadata={'input_tokens': 358, 'output_tokens': 196, 'total_tokens': 554, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reas

C:\Users\ansgy\AppData\Local\Temp\ipykernel_22192\4084101035.py:50: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  **docs[result.index].dict(),


{'messages': [HumanMessage(content='마이클 조던과 생일이 같은 유명인 3명 찾아줘', additional_kwargs={}, response_metadata={}, id='b09b35d8-4544-4085-bed0-9c29b351ac75'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"user_request": "\\ub9c8\\uc774\\ud074 \\uc870\\ub358 \\uc0dd\\uc77c"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--95a1c61a-85aa-4b89-8cd5-d08b3e4b03f7-0', tool_calls=[{'name': 'tavily_search_tool', 'args': {'user_request': '마이클 조던 생일'}, 'id': '899518a6-b612-4beb-92f6-52eb18bea104', 'type': 'tool_call'}], usage_metadata={'input_tokens': 358, 'output_tokens': 196, 'total_tokens': 554, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 169}}), ToolMessage(content="content='마이클 조던의 생일은 1963년 2월 17일입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'compl

C:\Users\ansgy\AppData\Local\Temp\ipykernel_22192\4084101035.py:50: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  **docs[result.index].dict(),


{'messages': [HumanMessage(content='마이클 조던과 생일이 같은 유명인 3명 찾아줘', additional_kwargs={}, response_metadata={}, id='b09b35d8-4544-4085-bed0-9c29b351ac75'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"user_request": "\\ub9c8\\uc774\\ud074 \\uc870\\ub358 \\uc0dd\\uc77c"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--95a1c61a-85aa-4b89-8cd5-d08b3e4b03f7-0', tool_calls=[{'name': 'tavily_search_tool', 'args': {'user_request': '마이클 조던 생일'}, 'id': '899518a6-b612-4beb-92f6-52eb18bea104', 'type': 'tool_call'}], usage_metadata={'input_tokens': 358, 'output_tokens': 196, 'total_tokens': 554, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 169}}), ToolMessage(content="content='마이클 조던의 생일은 1963년 2월 17일입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'compl

C:\Users\ansgy\AppData\Local\Temp\ipykernel_22192\4084101035.py:50: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  **docs[result.index].dict(),


{'messages': [HumanMessage(content='마이클 조던과 생일이 같은 유명인 3명 찾아줘', additional_kwargs={}, response_metadata={}, id='b09b35d8-4544-4085-bed0-9c29b351ac75'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"user_request": "\\ub9c8\\uc774\\ud074 \\uc870\\ub358 \\uc0dd\\uc77c"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--95a1c61a-85aa-4b89-8cd5-d08b3e4b03f7-0', tool_calls=[{'name': 'tavily_search_tool', 'args': {'user_request': '마이클 조던 생일'}, 'id': '899518a6-b612-4beb-92f6-52eb18bea104', 'type': 'tool_call'}], usage_metadata={'input_tokens': 358, 'output_tokens': 196, 'total_tokens': 554, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 169}}), ToolMessage(content="content='마이클 조던의 생일은 1963년 2월 17일입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'compl

C:\Users\ansgy\AppData\Local\Temp\ipykernel_22192\4084101035.py:50: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  **docs[result.index].dict(),


{'messages': [HumanMessage(content='마이클 조던과 생일이 같은 유명인 3명 찾아줘', additional_kwargs={}, response_metadata={}, id='b09b35d8-4544-4085-bed0-9c29b351ac75'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search_tool', 'arguments': '{"user_request": "\\ub9c8\\uc774\\ud074 \\uc870\\ub358 \\uc0dd\\uc77c"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--95a1c61a-85aa-4b89-8cd5-d08b3e4b03f7-0', tool_calls=[{'name': 'tavily_search_tool', 'args': {'user_request': '마이클 조던 생일'}, 'id': '899518a6-b612-4beb-92f6-52eb18bea104', 'type': 'tool_call'}], usage_metadata={'input_tokens': 358, 'output_tokens': 196, 'total_tokens': 554, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 169}}), ToolMessage(content="content='마이클 조던의 생일은 1963년 2월 17일입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'compl